# DAC Modular Pipeline Test

This notebook tests the complete EnCodec pipeline broken down into modular steps:

**Pipeline:**
1. `audio → latent_cont` (continuous embeddings before quantization)
2. `latent_cont → codes` (quantization to discrete codes)
3. `codes → latent_quant` (dequantize codes back to embeddings)
4. `latent_quant → audio` (decode quantized embeddings to audio)
5. `latent_cont → audio` (decode continuous embeddings directly)

**Comparisons:**
- Original audio
- Audio from continuous latents (no quantization)
- Audio from quantized latents (via codes path)

We'll test both 24kHz and 44.1kHz models using the guitar sample.

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent 
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

import torch
from IPython.display import Audio, display
import librosa

# Import our DAC processor
from processors.wrapper_dac import DACProcessor

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## **1. Load 24kHz model and audio in such sampling rate**

In [ ]:
sr = 24000

# Initialize 24kHz processor
processor_24k = DACProcessor(sr=sr)
print(f"\nModel info:")
info = processor_24k.get_model_info()
for key, value in info.items():
    print(f"  {key}: {value}")

audio_path = "../samples/guitar.wav"
audio_input, _ = librosa.load(audio_path, sr=sr, mono=True)
audio_input = torch.tensor(audio_input).unsqueeze(0)  # Add batch dimension
audio_input = audio_input[:, :sr*10]  # First 10 seconds

# Listen to original audio
print("\nOriginal Sound of shape ", audio_input.shape)
display(Audio(audio_input.numpy(), rate=sr))

In [ ]:
# Step 1: audio → latent_cont (continuous embeddings)
print("\n1️⃣ Audio → Continuous Latent Embeddings")
latent_cont, latent_cont_meta = processor_24k.audio_to_latents(audio_input, sr)
print(f"   Input audio shape: {audio_input.shape}")
print(f"   Number of latent chunks: {len(latent_cont)}")
print(f"   First chunk shape: {latent_cont[0].shape}")
print(f"   Metadata: {latent_cont_meta}")

In [ ]:
# Step 2: latent_cont → codes (quantization)
print("\n2️⃣ Continuous Latents → Discrete Codes")
codes_24k, codes_metadata_24k = processor_24k.latents_to_codes(latent_cont, latents_meta=latent_cont_meta)
print(f"   Discrete codes shape: {codes_24k.shape}")
print(f"   Codes metadata: {codes_metadata_24k}")
print(f"   Code values range: [{codes_24k.min().item()}, {codes_24k.max().item()}]")

In [ ]:
# Step 3: codes → latent_quant (quantized embeddings)
print("\n3️⃣ Discrete Codes → Quantized Latent Embeddings")
latent_quant_24k = processor_24k.codes_to_latents(codes_24k)
print(f"   Quantized latent embeddings shape: {latent_quant_24k.shape}")
print(f"   Embedding values range: [{latent_quant_24k.min().item():.3f}, {latent_quant_24k.max().item():.3f}]")

In [ ]:
# Step 4a: latent_quant → audio (from quantized embeddings)
print("\n4️⃣a Quantized Latents → Audio")
audio_from_latent_quant_24k = processor_24k.decode_latents_audio(latent_quant_24k, codes_metadata_24k)
print(f"   Audio from quantized latents shape: {audio_from_latent_quant_24k.shape}")

# Step 4b: latent_cont → audio (from continuous embeddings)
print("\n4️⃣b Continuous Latents → Audio")
audio_from_latent_cont_24k = processor_24k.decode_latents_audio(latent_cont)
print(f"   Audio from continuous latents shape: {audio_from_latent_cont_24k.shape}")

In [ ]:
# Step 5: Compare three audio reconstructions
print("\n🎵 24kHz Model Audio Comparisons:")

# Get equal length samples for comparison
min_len = min(audio_input.shape[-1], 
              audio_from_latent_cont_24k.shape[-1],
              audio_from_latent_quant_24k.shape[-1])

orig_24k = audio_input[..., :min_len]
cont_24k = audio_from_latent_cont_24k[..., :min_len]
quant_24k = audio_from_latent_quant_24k[..., :min_len]

print("\n✨ Original Audio:")
display(Audio(orig_24k.squeeze().cpu().numpy(), rate=24000))

print("\n🔵 Audio reconstructed from Continuous Latents:")
display(Audio(cont_24k.squeeze().cpu().numpy(), rate=24000))

print("\n🟢 Audio reconstructed from Quantized Latents (via codes):")
display(Audio(quant_24k.squeeze().cpu().numpy(), rate=24000))

In [ ]:
# Average stereo to mono for fair comparison
cont_24k_mono  = cont_24k.mean(dim=1, keepdim=True) if cont_24k.shape[1] == 2 else cont_24k
quant_24k_mono = quant_24k.mean(dim=1, keepdim=True) if quant_24k.shape[1] == 2 else quant_24k
cont_24k_mono = cont_24k_mono[0].to(device=orig_24k.device)
quant_24k_mono = quant_24k_mono[0].to(device=orig_24k.device)

# Shapes 
print(f"Original (mono): {orig_24k.shape}")
print(f"Continuous path (mono): {cont_24k_mono.shape}")
print(f"Quantized path (mono): {quant_24k_mono.shape}")

print("\n📊 MSE Analysis:")
mse_cont = torch.nn.functional.mse_loss(cont_24k_mono, orig_24k)
print(f"MSE (Continuous path):  {mse_cont.item():.8f}")
mse_quant = torch.nn.functional.mse_loss(quant_24k_mono, orig_24k)
print(f"MSE (Quantized path):   {mse_quant.item():.8f}")

# Multiscale FFT Loss
from auxiliar.losses import MultiscaleSpectrogramLoss
mss_loss = MultiscaleSpectrogramLoss(scales=[4096, 2048, 1024, 512], 
                                    overlap=0.75)
spectral_loss_fn = lambda x,y: mss_loss(x.view(1,-1), y.view(1,-1))

print("\n📊 Multiscale FFT Analysis:")
# Compute FFTs on mono versions
loss_cont  = spectral_loss_fn(orig_24k, cont_24k_mono)
loss_quant = spectral_loss_fn(orig_24k, quant_24k_mono)
print(f"Multiscale Spectral Loss (Continuous path): {loss_cont.item():.8f}")
print(f"Multiscale Spectral Loss (Quantized path):  {loss_quant.item():.8f}")

## **2. Load 48kHz model and audio in such sampling rate**

In [ ]:
sr = 44100

# Initialize 44.1kHz processor
processor_44k = DACProcessor(sr=sr)
print(f"\nModel info:")
info = processor_44k.get_model_info()
for key, value in info.items():
    print(f"  {key}: {value}")

audio_path = "../samples/guitar.wav"
audio_input, _ = librosa.load(audio_path, sr=sr, mono=True)
audio_input = torch.tensor(audio_input).unsqueeze(0)  # Add batch dimension
audio_input = audio_input[:, :sr*10]

# Listen to original audio
print("\nOriginal Sound of shape ", audio_input.shape)
display(Audio(audio_input.numpy(), rate=sr))

In [ ]:
# Step 1: audio → latent_cont (continuous embeddings)
print("\n1️⃣ Audio → Continuous Latent Embeddings")
latent_cont, latent_cont_meta = processor_44k.audio_to_latents(audio_input, sr)
print(f"   Input audio shape: {audio_input.shape}")
print(f"   Number of latent chunks: {len(latent_cont)}")
print(f"   First chunk shape: {latent_cont[0].shape}")
print(f"   Metadata keys: {list(latent_cont_meta.keys()) if latent_cont_meta else 'None'}")

In [ ]:
# Step 2: latent_cont → codes (quantization)
print("\n2️⃣ Continuous Latents → Discrete Codes")
codes, codes_metadata = processor_44k.latents_to_codes(latent_cont, latents_meta=latent_cont_meta)
print(f"   Discrete codes shape: {codes.shape}")
print(f"   Codes metadata keys: {list(codes_metadata.keys()) if codes_metadata else 'None'}")
print(f"   Code values range: [{codes.min().item()}, {codes.max().item()}]")
print(f"   Number of codebooks: {codes.shape[1]}")

In [ ]:
# Step 3: codes → latent_quant (quantized embeddings)
print("\n3️⃣ Discrete Codes → Quantized Latent Embeddings")
latent_quant = processor_44k.codes_to_latents(codes)
print(f"   Quantized latent embeddings shape: {latent_quant.shape}")
print(f"   Embedding values range: [{latent_quant.min().item():.3f}, {latent_quant.max().item():.3f}]")

In [ ]:
# Step 4a: latent_quant → audio (from quantized embeddings)
print("\n4️⃣a Quantized Latents → Audio")
audio_from_latent_quant = processor_44k.decode_latents_audio(latent_quant)
print(f"   Audio from quantized latents shape: {audio_from_latent_quant.shape}")

# Step 4b: latent_cont → audio (from continuous embeddings)
print("\n4️⃣b Continuous Latents → Audio")
audio_from_latent_cont = processor_44k.decode_latents_audio(latent_cont)
print(f"   Audio from continuous latents shape: {audio_from_latent_cont.shape}")

In [ ]:
# Step 5: Compare three audio reconstructions
print("\n🎵 44.1kHz DAC Model Audio Comparisons:")

# Get equal length samples for comparison
min_len = min(audio_input.shape[-1],
              audio_from_latent_cont.shape[-1],
              audio_from_latent_quant.shape[-1])

orig = audio_input[..., :min_len]
cont = audio_from_latent_cont[..., :min_len]
quant = audio_from_latent_quant[..., :min_len]

print("\n✨ Original Audio:")
display(Audio(orig.squeeze().cpu().numpy(), rate=sr))

print("\n🔵 Audio reconstructed from Continuous Latents:")
display(Audio(cont.squeeze().cpu().numpy(), rate=sr))

print("\n🟢 Audio reconstructed from Quantized Latents (via codes):")
display(Audio(quant.squeeze().cpu().numpy(), rate=sr))

In [ ]:
# Average stereo to mono for fair comparison
cont_44k_mono = cont.mean(dim=1, keepdim=True) if cont.shape[1] == 2 else cont
quant_44k_mono = quant.mean(dim=1, keepdim=True) if quant.shape[1] == 2 else quant
cont_44k_mono = cont_44k_mono[0].to(device=orig.device)
quant_44k_mono = quant_44k_mono[0].to(device=orig.device)

# Check shapes
print(f"Original 44.1kHz audio shape: {orig.shape}")
print(f"Continuous path audio shape: {cont_44k_mono.shape}")
print(f"Quantized path audio shape: {quant_44k_mono.shape}")

print("\n📊 MSE Analysis:")
mse_cont_44k = torch.nn.functional.mse_loss(cont_44k_mono, orig)
print(f"MSE (Continuous path):  {mse_cont_44k.item():.8f}")
mse_quant_44k = torch.nn.functional.mse_loss(quant_44k_mono, orig)
print(f"MSE (Quantized path):   {mse_quant_44k.item():.8f}")

# Multiscale FFT Loss
from auxiliar.losses import MultiscaleSpectrogramLoss
mss_loss = MultiscaleSpectrogramLoss(scales=[8192, 4096, 2048, 1024], 
                                    overlap=0.75)
spectral_loss_fn = lambda x,y: mss_loss(x.view(1,-1), y.view(1,-1))

print("\n📊 Multiscale FFT Analysis:")
loss_cont_44k  = spectral_loss_fn(orig, cont_44k_mono)
loss_quant_44k = spectral_loss_fn(orig, quant_44k_mono)
print(f"Multiscale Spectral Loss (Continuous path): {loss_cont_44k.item():.8f}")
print(f"Multiscale Spectral Loss (Quantized path):  {loss_quant_44k.item():.8f}")